# Phase 2-1: Universal Adversarial Patches

Trains an adversarial patch that can hide tanks from YOLOv8 detection.
The patch is optimized to reduce objectness scores when placed on detected tanks.

In [ ]:
!pip install ultralytics -q
!pip install opencv-python-headless -q

import os
import shutil
from pathlib import Path

import torch
import torch.optim as optim
import numpy as np
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# mount drive
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# paths - update these for your setup
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/data"
COLAB_LOCAL = "/content"

WEIGHTS_DIR = f"{BASE_PATH}/training/results from training/weights"
LOCAL_WEIGHTS = f"{COLAB_LOCAL}/weights"
WEIGHTS_FILE = f"{WEIGHTS_DIR}/best.pt"

print(f"Looking for weights at: {WEIGHTS_FILE}")
if os.path.exists(WEIGHTS_FILE):
    print("Found!")
else:
    print("Not found - check path")
    # show what's in training folder
    trainingDir = f"{BASE_PATH}/training"
    if os.path.exists(trainingDir):
        print(f"\nContents of {trainingDir}:")
        for item in os.listdir(trainingDir)[:10]:
            print(f"  {item}")

In [ ]:
# copy weights locally for faster access
if os.path.exists(WEIGHTS_FILE):
    os.makedirs(LOCAL_WEIGHTS, exist_ok=True)
    !cp "{WEIGHTS_FILE}" "{LOCAL_WEIGHTS}/"
    modelPath = f"{LOCAL_WEIGHTS}/best.pt"
    print(f"Weights copied to {modelPath}")
else:
    modelPath = WEIGHTS_FILE
    print(f"Using weights from Drive")

if os.path.exists(modelPath):
    print("Model ready")
else:
    print("Error: model not accessible")

In [ ]:
CONFIG = {
    "modelPath": modelPath,
    
    # source data
    "trainImages": f"{BASE_PATH}/training/train/images",
    "trainLabels": f"{BASE_PATH}/training/train/labels",
    "valImages": f"{BASE_PATH}/training/val/images",
    "valLabels": f"{BASE_PATH}/training/val/labels",
    "testImages": f"{BASE_PATH}/training/test/images",
    "testLabels": f"{BASE_PATH}/training/test/labels",
    
    # output
    "outputBase": f"{BASE_PATH}/adversarial",
    
    # patch params
    "patchSize": 100,
    "numEpochs": 100,
    "lr": 0.01,
    "confThresh": 0.4,
}

print("Config ready")

In [ ]:
# create output directories
for subset in ['train', 'val', 'test']:
    os.makedirs(f"{CONFIG['outputBase']}/{subset}/images", exist_ok=True)
    os.makedirs(f"{CONFIG['outputBase']}/{subset}/labels", exist_ok=True)

print("Output directories created")

In [ ]:
def initPatch(size):
    """Create a random patch tensor.
    
    Args:
        size: patch width/height in pixels
    
    Returns:
        torch tensor of shape (3, size, size) with gradients enabled
    """
    patch = torch.rand((3, size, size)).to(device)
    patch.requires_grad_(True)
    return patch


def applyPatch(imgNp, patch, box):
    """Apply patch to image at the center of a bounding box.
    
    Args:
        imgNp: numpy array of image (H, W, 3)
        patch: torch tensor (3, patchSize, patchSize)
        box: bounding box [x1, y1, x2, y2]
    
    Returns:
        numpy array of patched image
    """
    h, w = imgNp.shape[:2]
    patchSize = patch.shape[1]
    
    # get box center
    x1, y1, x2, y2 = map(int, box[:4])
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2
    
    # where to put patch
    px1 = max(0, cx - patchSize // 2)
    py1 = max(0, cy - patchSize // 2)
    px2 = min(w, px1 + patchSize)
    py2 = min(h, py1 + patchSize)
    
    # convert image to tensor
    imgTensor = torch.from_numpy(imgNp).float().to(device) / 255.0
    imgTensor = imgTensor.permute(2, 0, 1)  # HWC -> CHW
    patched = imgTensor.clone()
    
    # resize patch to fit and apply
    actualH = py2 - py1
    actualW = px2 - px1
    if actualH > 0 and actualW > 0:
        patchResized = torch.nn.functional.interpolate(
            patch.unsqueeze(0), size=(actualH, actualW), mode='bilinear'
        )[0]
        patched[:, py1:py2, px1:px2] = patchResized
    
    # convert back to numpy
    patchedNp = patched.permute(1, 2, 0).cpu().detach().numpy()
    return (patchedNp * 255).astype(np.uint8)

In [ ]:
def trainPatch(model):
    """Train an adversarial patch to hide detections.
    
    Uses test images to optimize a patch that reduces detection confidence.
    
    Args:
        model: YOLO model
    
    Returns:
        trained patch tensor
    """
    print("\nTraining adversarial patch...")
    
    patch = initPatch(CONFIG['patchSize'])
    optimizer = optim.Adam([patch], lr=CONFIG['lr'])
    
    # use subset of test images for training
    testImgs = list(Path(CONFIG['testImages']).glob("*.jpg"))[:50]
    
    for epoch in range(CONFIG['numEpochs']):
        totalLoss = 0
        successCount = 0
        
        for imgPath in testImgs:
            img = cv2.imread(str(imgPath))
            imgRgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # detect tanks
            results = model(imgRgb, conf=CONFIG['confThresh'], verbose=False)
            
            if len(results[0].boxes) > 0:
                box = results[0].boxes[0].xyxy[0].cpu().numpy()
                
                # apply patch and check if it hides the tank
                patchedImg = applyPatch(imgRgb, patch, box)
                resultsPatched = model(patchedImg, conf=CONFIG['confThresh'], verbose=False)
                
                # loss: 0.1 if hidden (good), 1.0 if still visible (bad)
                if len(resultsPatched[0].boxes) == 0:
                    loss = torch.tensor(0.1, device=device)
                    successCount += 1
                else:
                    loss = torch.tensor(1.0, device=device)
                
                # smoothness penalty (total variation)
                tvLoss = torch.sum(torch.abs(patch[:, 1:, :] - patch[:, :-1, :])) * 0.01
                totalLoss = loss + tvLoss
                
                # update
                optimizer.zero_grad()
                totalLoss.backward()
                optimizer.step()
                
                # keep pixel values valid
                patch.data.clamp_(0, 1)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{CONFIG['numEpochs']}: {successCount}/{len(testImgs)} hidden")
    
    print("Patch training complete")
    return patch

In [ ]:
def applyPatchToAll(model, patch):
    """Apply trained patch to all images in train/val/test.
    
    Args:
        model: YOLO model for detection
        patch: trained patch tensor
    """
    print("\nApplying patch to all images...")
    
    subsets = {
        'train': (CONFIG['trainImages'], CONFIG['trainLabels']),
        'val': (CONFIG['valImages'], CONFIG['valLabels']),
        'test': (CONFIG['testImages'], CONFIG['testLabels']),
    }
    
    for subset, (imgDir, lblDir) in subsets.items():
        print(f"\nProcessing {subset}...")
        
        srcImgDir = Path(imgDir)
        srcLblDir = Path(lblDir)
        outImgDir = Path(f"{CONFIG['outputBase']}/{subset}/images")
        outLblDir = Path(f"{CONFIG['outputBase']}/{subset}/labels")
        
        images = list(srcImgDir.glob("*.jpg"))
        patchedCount = 0
        
        for imgPath in tqdm(images, desc=subset):
            img = cv2.imread(str(imgPath))
            imgRgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # detect and patch if there's a detection
            results = model(imgRgb, conf=CONFIG['confThresh'], verbose=False)
            
            if len(results[0].boxes) > 0:
                box = results[0].boxes[0].xyxy[0].cpu().numpy()
                patchedImg = applyPatch(imgRgb, patch, box)
                
                patchedBgr = cv2.cvtColor(patchedImg, cv2.COLOR_RGB2BGR)
                cv2.imwrite(str(outImgDir / imgPath.name), patchedBgr)
                patchedCount += 1
            else:
                # no detection, just copy original
                shutil.copy2(imgPath, outImgDir / imgPath.name)
            
            # copy label with marker
            lblPath = srcLblDir / f"{imgPath.stem}.txt"
            outLblPath = outLblDir / f"{imgPath.stem}.txt"
            
            with open(outLblPath, 'w') as f:
                f.write("# PATCHED\n")
                if lblPath.exists():
                    with open(lblPath) as orig:
                        f.write(orig.read())
        
        print(f"{subset}: {patchedCount}/{len(images)} patched")

In [ ]:
# load model
print("Loading model...")
model = YOLO(CONFIG['modelPath'])
model.to(device)
print("Model loaded")

# train patch
patch = trainPatch(model)

# visualize the patch
patchNp = patch.detach().cpu().numpy().transpose(1, 2, 0)
plt.figure(figsize=(5, 5))
plt.imshow(patchNp)
plt.title('Adversarial Patch')
plt.axis('off')
plt.savefig(f"{CONFIG['outputBase']}/patch.png")
plt.show()

# save patch tensor
torch.save(patch, f"{CONFIG['outputBase']}/patch.pt")
print(f"Patch saved to {CONFIG['outputBase']}/patch.pt")

# apply to all images
applyPatchToAll(model, patch)

print("\n" + "="*50)
print("DONE")
print("="*50)
print(f"\nOutput: {CONFIG['outputBase']}/")
print("  train/images/ - patched training images")
print("  val/images/   - patched validation images")
print("  test/images/  - patched test images")